## Link selection app from a scraped list of links from a URL
1. Set up the system prompt "link system prompt"
2. use the fetch website link function to get links from a URL and feed it to the user prompt of which it's part of to select the relevent URLs
3. use a function which will feed both the system prompt and the user prompt to the chat completions.
4. 

In [4]:
from openai import OpenAI
from dotenv import load_dotenv
load_dotenv()
import json
from IPython.display import Markdown, display
openai = OpenAI()

from scraper import fetch_website_links


In [5]:
#Set up the system prompt "link system prompt"
# prompt to the LLM to select based on example ( one shot prompt )
link_system_prompt = """
You are provided with a list of links found on a webpage.
You are able to decide which of the links would be most relevant to include in a brochure about the company,
such as links to an About page, or a Company page, or Careers/Jobs pages.
You should respond in JSON as in this example:

{
    "links": [
        {"type": "about page", "url": "https://full.url/goes/here/about"},
        {"type": "careers page", "url": "https://another.full.url/careers"}
        {"type": "page title", "summary": "summary of the URL contents" }
    ]
}
"""




In [6]:
# use the fetch website link function to get links from a URL and feed it to the user prompt of which it's part of to select the relevent URLs
def get_links_user_prompt(url):
    user_prompt = f"""
Here is the list of links on the website {url} -
Please decide which of these are relevant web links for a brochure about the company, 
respond with the full https URL in JSON format.
Do not include Terms of Service, Privacy, email links.
give a brief summary of what the link contents are.

Links (some might be relative links):

"""
    links = fetch_website_links(url)
    user_prompt += "\n".join(links)
    return user_prompt

In [7]:
# Use a function which will feed both the system prompt and the user prompt to the chat completions.
def select_relevant_links(url):
    response = openai.chat.completions.create(
        model="gpt-4o-mini",
        messages=[
            {"role": "system", "content": link_system_prompt},
            {"role": "user", "content": get_links_user_prompt(url)}
        ],
        response_format={"type": "json_object"}
    )
    result = response.choices[0].message.content
    links = json.loads(result)
    return links

In [8]:
select_relevant_links("https://www.databricks.com")

{'links': [{'type': 'about page',
   'url': 'https://www.databricks.com/company/about-us',
   'summary': 'This page provides an overview of Databricks, its mission, vision, and values, detailing its role in the data and AI landscape.'},
  {'type': 'leadership team page',
   'url': 'https://www.databricks.com/company/leadership-team',
   'summary': "This page lists the members of Databricks' leadership team, showcasing their backgrounds and roles within the company."},
  {'type': 'careers page',
   'url': 'https://www.databricks.com/company/careers',
   'summary': 'The careers page highlights job opportunities at Databricks, along with information about company culture and employee benefits.'},
  {'type': 'awards and recognition page',
   'url': 'https://www.databricks.com/company/awards-and-recognition',
   'summary': 'This page showcases the major awards and recognition Databricks has received for its services and innovations in the field of data and AI.'},
  {'type': 'newsroom page',

In [14]:
def build_brochure(url):
  response = openai.chat.completions.create(model="gpt-4o-mini",
  messages=[
    {"role" : "system", "content" : link_system_prompt},
    {"role" : "user", "content" : get_links_user_prompt(url) }
  ]
  )
  brochure = response.choices[0].message.content
  return display(Markdown(brochure))

In [15]:
brochure_pdct = build_brochure("https://www.databricks.com")
brochure_pdct

```json
{
    "links": [
        {
            "type": "about page",
            "url": "https://www.databricks.com/company/about-us",
            "summary": "Provides an overview of Databricks, its mission, vision, and values."
        },
        {
            "type": "leadership team page",
            "url": "https://www.databricks.com/company/leadership-team",
            "summary": "Details the leadership team at Databricks, highlighting key executives and their backgrounds."
        },
        {
            "type": "careers page",
            "url": "https://www.databricks.com/company/careers",
            "summary": "Offers information about job opportunities at Databricks and the company's culture."
        },
        {
            "type": "open positions page",
            "url": "https://www.databricks.com/company/careers/open-positions",
            "summary": "Lists current job openings available within Databricks across various departments."
        },
        {
            "type": "awards and recognition page",
            "url": "https://www.databricks.com/company/awards-and-recognition",
            "summary": "Showcases the accolades and recognitions received by Databricks in the industry."
        },
        {
            "type": "newsroom page",
            "url": "https://www.databricks.com/company/newsroom",
            "summary": "Provides access to press releases and news articles about Databricks."
        }
    ]
}
```